# Belgian Real Estate — ML Price Prediction Pipeline

This notebook implements the **complete machine learning pipeline** for predicting Belgian real estate prices, from cleaned data to serialized production models.

## Pipeline Overview

| Step | Description | Key function(s) |
|------|-------------|-----------------|
| 1 | Data loading & initial cleaning | `first_cleaning()` |
| 2 | Split dataset by property type | `split_property_type()` |
| 3 | Outlier removal & feature pruning | `clean_outliers()`, `remove_unused_columns()` |
| 4 | ML preprocessing (encoding, scaling) | `ML_preprocessing()` |
| 5 | Feature selection (VT + Lasso) | `feature_selection()` |
| 6 | Model training & cross-validation | `best_models()` |

## Input
- `data/processed/cleaned_properties_dataset_2026-03-12-13h12.csv` — canonical cleaned dataset (13,099 rows)

## Output
- `deployment/api/models/best_model_houses.pkl`
- `deployment/api/models/best_model_apartments.pkl`
- `deployment/api/models/scaler_houses.pkl`
- `deployment/api/models/scaler_apartments.pkl`

---
## Imports

In [ ]:
# Data
import pandas as pd
import numpy as np

# ML — preprocessing, transformation, regression
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.feature_selection import VarianceThreshold, SelectKBest, mutual_info_regression, SelectFromModel
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
import joblib

---
## Step 1 — Data Loading & Initial Cleaning

Loads the canonical cleaned dataset and applies a final round of adjustments before ML:

- Remove duplicate rows
- Drop rows with missing `price` or `living_area`
- Strip leading/trailing whitespace from string columns
- Fix specific city name edge case (`"La"` → `"La Louvière"`)
- Synchronize boolean flags with their area counterparts:
  - If `garden_area > 0` → `garden = True`; if `garden = False` → `garden_area = 0`
  - Same logic for `terrace` / `terrace_area`
- Fill missing boolean flags (`swimming_pool`, `open_fire`, `furnished`) with `False`
- Replace `living_area == 0` with `NaN`
- Drop properties priced under €10,000
- Normalize column types (`zip_code` → str, float columns → Int64)

In [ ]:
def first_cleaning(path: str) -> pd.DataFrame:
    """
    Loads the processed dataset and applies final pre-ML cleaning adjustments.

    Args:
        path (str): Path to the canonical cleaned CSV file.

    Returns:
        pd.DataFrame: A cleaned DataFrame ready for splitting and ML preprocessing.
    """
    properties = pd.read_csv(path, index_col=0)

    # Remove duplicates
    properties = properties.drop_duplicates().reset_index(drop=True)

    # Drop rows with missing target or key feature
    empty_lines_price = properties[properties['price'].isna()].index
    empty_lines_area = properties[properties['living_area'].isna()].index
    empty_lines = empty_lines_price.append(empty_lines_area)
    properties = properties.drop(empty_lines, axis=0)

    # Strip whitespace from all string columns
    properties = properties.apply(lambda x: x.str.strip() if x.dtype == "object" else x)

    # Fix known city name parsing edge case
    properties.loc[properties.city == 'La', 'city'] = "La Louvière"

    # Synchronize garden flag and area
    properties.loc[properties['garden_area'] > 0, 'garden'] = True
    properties.loc[
        (properties['garden'] == True) &
        ((properties['garden_area'] == 0) | (properties['garden_area'].isna())),
        'garden_area'
    ] = np.nan
    properties.loc[properties['garden'] == False, 'garden_area'] = 0

    # Synchronize terrace flag and area
    properties.loc[properties['terrace_area'] > 0, 'terrace'] = True
    properties.loc[
        (properties['terrace'] == True) &
        ((properties['terrace_area'] == 0) | (properties['terrace_area'].isna())),
        'terrace_area'
    ] = np.nan
    properties.loc[properties['terrace'] == False, 'terrace_area'] = 0

    # Fill missing boolean flags with False
    properties['garden'] = properties['garden'].fillna(False)
    properties['terrace'] = properties['terrace'].fillna(False)
    properties['swimming_pool'] = properties['swimming_pool'].fillna(False)
    properties['open_fire'] = properties['open_fire'].fillna(False)
    properties['furnished'] = properties['furnished'].fillna(False)

    # Replace zero living area with NaN (invalid data)
    properties.loc[properties['living_area'] == 0, 'living_area'] = np.nan

    # Remove clearly erroneous prices
    properties = properties[properties['price'] >= 10000]

    # Normalize types: zip_code as string, floats as nullable Int64
    properties["zip_code"] = properties["zip_code"].astype(int).astype(str)
    float_cols = properties.select_dtypes('float64').columns
    properties[float_cols] = properties[float_cols].astype("Int64")

    properties = properties.drop_duplicates()

    return properties

---
## Step 2 — Data Splitting

Splits the dataset into two independent subsets for separate model training:
- **Houses**: residence, villa, cottage, bungalow, mansion, etc.
- **Apartments**: apartment, studio, duplex, penthouse, loft, etc.

Training separate models per property type significantly improves accuracy, as houses and
apartments have very different feature distributions (e.g., land surface, number of facades).

`split_by_region()` is provided as an optional utility for geographic sub-analysis.

In [ ]:
def split_property_type(properties: pd.DataFrame):
    """
    Splits the dataset into houses and apartments subsets.

    Args:
        properties (pd.DataFrame): The full cleaned dataset.

    Returns:
        tuple: (houses DataFrame, apartments DataFrame)
    """
    houses = properties[properties.type_of_property == 'House'].reset_index(drop=True)
    apartments = properties[properties.type_of_property == 'Apartment'].reset_index(drop=True)
    return houses, apartments


def split_by_region(properties: pd.DataFrame):
    """
    Splits the dataset by Belgian administrative region.

    Args:
        properties (pd.DataFrame): Dataset with a 'region' column.

    Returns:
        tuple: (wallonia, flanders, brussels) DataFrames
    """
    wallonia = properties[properties.region == 'Wallonia'].reset_index(drop=True)
    flanders = properties[properties.region == 'Flanders'].reset_index(drop=True)
    brussels = properties[properties.region == 'Brussels'].reset_index(drop=True)
    return wallonia, flanders, brussels

---
## Step 3 — Outlier Removal & Feature Pruning

### Outlier Removal
Uses the **IQR method** with a configurable whisker multiplier (default: `2×IQR`, softer than the standard `1.5×IQR`).

Columns checked: `price`, `number_of_rooms`, `living_area`, `garden_area` (for properties that have a garden).

A wider whisker is chosen deliberately to preserve more price diversity while removing only the most extreme values.

### Feature Pruning
Drops columns that are not used by the model:
- `type_of_property` — already split into separate DataFrames
- `url`, `zip_code`, `city` — identifiers, not predictive features
- `price_sqm` — derived from `price`, would cause data leakage
- `type_of_sale`, `subtype_of_property` — low signal for price regression
- Fully empty columns

In [ ]:
def clean_outliers(df: pd.DataFrame, whisker: float = 2) -> pd.DataFrame:
    """
    Removes statistical outliers using the IQR method.

    Args:
        df (pd.DataFrame): Input property DataFrame.
        whisker (float): IQR multiplier for the fence. Default is 2 (softer than standard 1.5).

    Returns:
        pd.DataFrame: DataFrame with outlier rows removed.
    """
    cols_to_clean = df[['price', 'number_of_rooms', 'living_area']]

    Q1 = cols_to_clean.quantile(0.25)
    Q3 = cols_to_clean.quantile(0.75)
    IQR = Q3 - Q1
    low_limit = (Q1 - whisker * IQR).clip(lower=0)
    high_limit = Q3 + whisker * IQR

    # Compute garden_area limit separately (only for properties with a garden)
    prop_with_garden = df.loc[df['garden_area'] > 0]
    garden_Q1 = prop_with_garden['garden_area'].quantile(0.25)
    garden_Q3 = prop_with_garden['garden_area'].quantile(0.75)
    garden_IQR = garden_Q3 - garden_Q1
    high_limit['garden_area'] = garden_Q3 + whisker * garden_IQR

    print("Upper limits used for outlier removal:")
    print(high_limit)

    # Build mask on NaN-replaced copy to avoid filtering out missing values
    sub = df.replace(np.nan, 0)
    col_names = cols_to_clean.columns
    mask = sub[col_names].apply(
        lambda x: x.between(low_limit[x.name], high_limit[x.name])
    ).all(axis=1)

    return df[mask]


def remove_unused_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Drops fully-empty columns and features not used by the ML model.

    Args:
        df (pd.DataFrame): Input DataFrame.

    Returns:
        pd.DataFrame: DataFrame with only model-relevant features.
    """
    df = df.dropna(axis=1, how='all')
    unused = ['type_of_property', 'url', 'zip_code', 'city',
              'price_sqm', 'type_of_sale', 'subtype_of_property']
    for col in unused:
        try:
            df = df.drop(col, axis=1)
        except KeyError:
            continue
    return df.reset_index(drop=True)

---
## Step 4 — ML Preprocessing

Transforms the cleaned DataFrame into a format suitable for scikit-learn / XGBoost:

1. **One-hot encoding** — `pd.get_dummies()` on all remaining categorical columns
   (e.g., `province_Brussels`, `region_Wallonia`, ...)
2. **Train/test split** — default 80/20 ratio, stratified by `random_state`
3. **Missing value imputation**:
   - `garden_area`: filled with the median area of properties **that have a garden**
   - `terrace_area`: same conditional logic
   - All other numeric columns: filled with column median
4. **Feature scaling** — `StandardScaler` fitted **only on the training set** to prevent
   data leakage, then applied to the test set via `transform()`

Returns the fitted scaler (`sc`) alongside the train/test arrays, so it can be
serialized and reused by the API for live predictions.

In [ ]:
def dummy_and_split(df: pd.DataFrame, test_size: float = 0.2, random_state: int = 42):
    """
    One-hot encodes categorical columns and splits into train/test sets.

    Args:
        df (pd.DataFrame): Input DataFrame with a 'price' target column.
        test_size (float): Proportion of data for the test set.
        random_state (int): Seed for reproducibility.

    Returns:
        tuple: (X_train, X_test, y_train, y_test)
    """
    df = pd.get_dummies(df, drop_first=True)
    X = df.drop(['price'], axis=1)
    y = df['price']
    return train_test_split(X, y, test_size=test_size, random_state=random_state)


def fill_nan(X: pd.DataFrame) -> pd.DataFrame:
    """
    Imputes missing values with column medians, using conditional medians
    for garden_area and terrace_area (only over rows where the feature exists).

    Args:
        X (pd.DataFrame): Feature matrix potentially containing NaN values.

    Returns:
        pd.DataFrame: DataFrame with all NaN values filled.
    """
    num_col = X.select_dtypes('Int64').columns

    if 'garden' in X.columns:
        median_garden = X.loc[X['garden'] == 1, 'garden_area'].median()
        X['garden_area'] = X['garden_area'].fillna(int(median_garden))
        num_col = num_col.drop(['garden_area'])

    if 'terrace' in X.columns:
        median_terrace = X.loc[X['terrace'] == 1, 'terrace_area'].median()
        X['terrace_area'] = X['terrace_area'].fillna(int(median_terrace))
        num_col = num_col.drop(['terrace_area'])

    X[num_col] = X[num_col].fillna(X[num_col].median().astype('Int64'))
    return X


def ML_preprocessing(df: pd.DataFrame, test_size: float = 0.2, random_state: int = 1):
    """
    Full ML preprocessing pipeline: encode → split → impute → scale.

    Args:
        df (pd.DataFrame): Cleaned property DataFrame with a 'price' target.
        test_size (float): Train/test split ratio.
        random_state (int): Seed for reproducibility.

    Returns:
        tuple: (X_train, X_test, y_train, y_test, feature_names, scaler)
    """
    X_train, X_test, y_train, y_test = dummy_and_split(df, test_size, random_state)

    X_train = fill_nan(X_train).astype('float64')
    X_test = fill_nan(X_test).astype('float64')
    y_train = y_train.astype('float64')
    y_test = y_test.astype('float64')

    feature_names = X_train.columns.tolist()

    # Fit scaler on training data only — prevents data leakage
    sc = StandardScaler()
    X_train = sc.fit_transform(X_train)
    X_test = sc.transform(X_test)

    return X_train, X_test, y_train, y_test, feature_names, sc

---
## Step 5 — Feature Selection

Reduces dimensionality in two stages:

1. **VarianceThreshold** (`threshold=0.01`) — removes near-constant features
   that carry no discriminative information.

2. **SelectFromModel with Lasso** (`α=10`) — applies L1 regularization;
   features whose coefficients are zeroed out by Lasso are dropped.
   A high alpha forces aggressive sparsity, keeping only the strongest predictors.

The final selected features per property type are printed for inspection.
These feature lists must match what the API's `prediction_service.py` expects.

In [ ]:
def feature_selection(X_train, X_test, y_train, feature_names: list):
    """
    Two-stage feature selection: VarianceThreshold then Lasso-based SelectFromModel.

    Args:
        X_train: Scaled training feature matrix.
        X_test: Scaled test feature matrix.
        y_train: Training target values.
        feature_names (list): Original feature names (pre-scaling).

    Returns:
        tuple: (X_train_selected, X_test_selected) after both selection stages.
    """
    # Stage 1: Variance Threshold — drop near-constant features
    vt = VarianceThreshold(threshold=0.01)
    X_train_vt = vt.fit_transform(X_train)
    X_test_vt = vt.transform(X_test)
    feature_names_vt = [feature_names[i] for i in vt.get_support(indices=True)]
    print(f"After VarianceThreshold: {feature_names_vt}")

    # Stage 2: Lasso-based selection — keep features with non-zero coefficients
    sfm = SelectFromModel(Lasso(alpha=10, max_iter=10000, random_state=0))
    X_train_sfm = sfm.fit_transform(X_train_vt, y_train)
    X_test_sfm = sfm.transform(X_test_vt)
    feature_names_sfm = [feature_names_vt[i] for i in sfm.get_support(indices=True)]
    print(f"After Lasso SelectFromModel: {feature_names_sfm}")

    return X_train_sfm, X_test_sfm

---
## Step 6 — Model Training

Trains two separate **XGBoost** regressors (one per property type).
Hyperparameters were selected through manual tuning:

| Property type | Learning rate | Estimators | Max depth | Subsample |
|---------------|:------------:|:----------:|:---------:|:---------:|
| Houses        | 0.02         | 380        | 5         | 0.8       |
| Apartments    | 0.03         | 250        | 6         | 0.6       |

Performance is measured with:
- **5-fold cross-validation R²** on the training set (variance check)
- **R² score** on the held-out test set
- **MAE** (Mean Absolute Error) — reported in euros after training

In [ ]:
def best_models(H_X_train, H_X_test, H_y_train, H_y_test,
                A_X_train, A_X_test, A_y_train, A_y_test):
    """
    Trains and evaluates XGBoost models for houses and apartments.

    Args:
        H_X_train, H_X_test, H_y_train, H_y_test: Houses train/test splits.
        A_X_train, A_X_test, A_y_train, A_y_test: Apartments train/test splits.

    Returns:
        tuple: (best_model_houses, best_model_apartments)
    """
    # Houses model
    best_model_houses = XGBRegressor(
        learning_rate=0.02, n_estimators=380, max_depth=5, subsample=0.8
    )
    cv_scores_h = cross_val_score(
        best_model_houses, H_X_train, H_y_train, cv=5, scoring='r2', n_jobs=-1
    )
    best_model_houses.fit(H_X_train, H_y_train)
    print("\n--- HOUSES ---")
    print(f"CV R² scores : {cv_scores_h}")
    print(f"Mean CV R²   : {cv_scores_h.mean():.3f} (±{cv_scores_h.std():.3f})")
    print(f"Test R²      : {best_model_houses.score(H_X_test, H_y_test):.3f}")

    # Apartments model
    best_model_apartments = XGBRegressor(
        learning_rate=0.03, n_estimators=250, max_depth=6, subsample=0.6
    )
    cv_scores_a = cross_val_score(
        best_model_apartments, A_X_train, A_y_train, cv=5, scoring='r2', n_jobs=-1
    )
    best_model_apartments.fit(A_X_train, A_y_train)
    print("\n--- APARTMENTS ---")
    print(f"CV R² scores : {cv_scores_a}")
    print(f"Mean CV R²   : {cv_scores_a.mean():.3f} (±{cv_scores_a.std():.3f})")
    print(f"Test R²      : {best_model_apartments.score(A_X_test, A_y_test):.3f}")

    return best_model_houses, best_model_apartments

---
## Pipeline Execution

Runs all steps end-to-end for both property types.

> **Note:** Paths are relative to this notebook's location (`notebooks/`).
> The dataset must be present at `data/processed/` before running.

In [ ]:
DATA_PATH = "../data/processed/cleaned_properties_dataset_2026-03-12-13h12.csv"

# Step 1 — Load and clean
all_properties = first_cleaning(DATA_PATH)

# Step 2 — Split by property type
houses, apartments = split_property_type(all_properties)

# Step 3 — Remove outliers and unused features
cleaned_houses = remove_unused_columns(clean_outliers(houses, whisker=2))
cleaned_apartments = remove_unused_columns(clean_outliers(apartments, whisker=2))

# Step 4 — ML preprocessing (encode, split, impute, scale)
H_X_train, H_X_test, H_y_train, H_y_test, H_feature_names, sc_house = ML_preprocessing(cleaned_houses)
A_X_train, A_X_test, A_y_train, A_y_test, A_feature_names, sc_apart = ML_preprocessing(cleaned_apartments)

# Step 5 — Feature selection
H_X_train_sfm, H_X_test_sfm = feature_selection(H_X_train, H_X_test, H_y_train, H_feature_names)
A_X_train_sfm, A_X_test_sfm = feature_selection(A_X_train, A_X_test, A_y_train, A_feature_names)

# Step 6 — Train models
best_model_houses, best_model_apartments = best_models(
    H_X_train_sfm, H_X_test_sfm, H_y_train, H_y_test,
    A_X_train_sfm, A_X_test_sfm, A_y_train, A_y_test
)

---
## Model Evaluation

**Mean Absolute Error (MAE)** on the held-out test set.
Represents the average prediction error in euros — easier to interpret than R².

Reference results from last training run:
- Houses: MAE ≈ €83,406
- Apartments: MAE ≈ €60,533

In [ ]:
H_y_hat = best_model_houses.predict(H_X_test_sfm)
H_MAE = mean_absolute_error(H_y_hat, H_y_test)

A_y_hat = best_model_apartments.predict(A_X_test_sfm)
A_MAE = mean_absolute_error(A_y_hat, A_y_test)

print(f"Houses     — MAE: €{H_MAE:,.0f}")
print(f"Apartments — MAE: €{A_MAE:,.0f}")

---
## Save Trained Models

Serializes the trained models and scalers to `deployment/api/models/` using `joblib`.

**Uncomment and run this cell only when satisfied with the evaluation results above.**
These files are gitignored — store them in a shared location or re-train from this notebook.

In [ ]:
MODELS_PATH = "../deployment/api/models/"

# joblib.dump(best_model_houses,    MODELS_PATH + "best_model_houses.pkl")
# joblib.dump(best_model_apartments, MODELS_PATH + "best_model_apartments.pkl")
# joblib.dump(sc_house,              MODELS_PATH + "scaler_houses.pkl")
# joblib.dump(sc_apart,              MODELS_PATH + "scaler_apartments.pkl")

# print("Models and scalers saved to", MODELS_PATH)